# Financial & ESG Forecasting

This notebook documents the final Random Forest forecasting stage of the
Campania Financial & ESG Forecasting project.

The original company-level dataset is proprietary and cannot be redistributed.
For this public portfolio, the notebook therefore combines:

1. the **original modelling design and reported evaluation metrics**; and
2. a **synthetic reproducibility demo** that mirrors the same forecasting logic.

After model benchmarking, Random Forest was retained with:

- `n_estimators = 100`
- `max_depth = 10`
- `min_samples_leaf = 10`

The model was then applied separately to the project's main economic and ESG
targets.

## 1. Forecasting tasks

The original analysis implemented four separate forecasting tasks:

| Target | Historical inputs | Predicted value |
|---|---|---|
| EBITDA | 2015–2023 | EBITDA 2024 |
| Sales Revenue | 2015–2023 | Sales Revenue 2024 |
| Net Income | 2015–2023 | Net Income 2024 |
| ESG score | 2019–2020 | ESG score 2021 |

Financial forecasts therefore use nine annual observations per company,
whereas the ESG model uses the two historical ESG observations available
before the 2021 target.

The same 75/25 train-test split was used, stratified by company size.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

## 2. Reported Random Forest evaluation

The following target-level MSE values are taken from the original analysis
notebook. They are reported here separately from the synthetic demonstration
below.

In [ ]:
reported_metrics = pd.DataFrame(
    {
        "Target": [
            "Sales Revenue",
            "EBITDA",
            "Net Income",
            "ESG",
        ],
        "MSE": [
            0.000713,
            0.002860,
            0.005480,
            0.004031,
        ],
    }
).sort_values("MSE")

reported_metrics

In [ ]:
ax = reported_metrics.plot(
    x="Target",
    y="MSE",
    kind="bar",
    legend=False,
    figsize=(8, 4),
)

ax.set_title("Random Forest — Reported Target-Level MSE")
ax.set_ylabel("Mean Squared Error")
ax.set_xlabel("")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 3. Reusable forecasting function

Each target is modelled independently using the same Random Forest
configuration selected during benchmarking.

In [ ]:
def fit_random_forest_forecast(
    dataframe: pd.DataFrame,
    feature_columns: list[str],
    target_column: str,
    stratify_column: str,
    random_state: int = 42,
):
    X = dataframe[feature_columns]
    y = dataframe[target_column]
    strata = dataframe[stratify_column]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=random_state,
        stratify=strata,
    )

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=10,
        random_state=random_state,
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    test_predictions = model.predict(X_test)
    full_predictions = model.predict(X)

    mse = mean_squared_error(y_test, test_predictions)

    return {
        "model": model,
        "mse": mse,
        "test_predictions": test_predictions,
        "full_predictions": full_predictions,
        "test_index": X_test.index,
    }

## 4. Synthetic public dataset

The following records are fully synthetic and do not correspond to real
companies. Their only purpose is to make the forecasting pipeline executable
without exposing proprietary data.

In [ ]:
rng = np.random.default_rng(42)

n_companies = 500

synthetic = pd.DataFrame(
    {
        "company_id": [f"COMP_{i:04d}" for i in range(n_companies)],
        "company_size": rng.choice(
            ["micro", "small", "medium/large"],
            size=n_companies,
            p=[0.35, 0.45, 0.20],
        ),
    }
)

def generate_financial_history(
    base_mean,
    base_sigma,
    growth_mean,
    growth_sigma,
    noise_scale,
):
    base = rng.lognormal(
        mean=base_mean,
        sigma=base_sigma,
        size=n_companies,
    )
    growth = rng.normal(
        loc=growth_mean,
        scale=growth_sigma,
        size=n_companies,
    )

    values = {}

    for step, year in enumerate(range(2015, 2025)):
        noise = rng.normal(
            loc=0.0,
            scale=noise_scale,
            size=n_companies,
        )
        values[year] = base * (1 + growth) ** step + noise

    return values


sales = generate_financial_history(
    base_mean=3.4,
    base_sigma=0.55,
    growth_mean=0.045,
    growth_sigma=0.025,
    noise_scale=0.50,
)

ebitda = generate_financial_history(
    base_mean=1.7,
    base_sigma=0.60,
    growth_mean=0.040,
    growth_sigma=0.035,
    noise_scale=0.18,
)

net_income = generate_financial_history(
    base_mean=1.2,
    base_sigma=0.70,
    growth_mean=0.030,
    growth_sigma=0.050,
    noise_scale=0.25,
)

for year in range(2015, 2025):
    synthetic[f"sales_{year}"] = sales[year]
    synthetic[f"ebitda_{year}"] = ebitda[year]
    synthetic[f"net_income_{year}"] = net_income[year]

esg_2019 = rng.beta(a=2.5, b=2.0, size=n_companies)
esg_2020 = np.clip(
    0.75 * esg_2019 + 0.25 * rng.beta(3.0, 2.0, size=n_companies),
    0,
    1,
)
esg_2021 = np.clip(
    0.75 * esg_2020 + 0.25 * rng.beta(3.2, 1.9, size=n_companies),
    0,
    1,
)

synthetic["esg_2019"] = esg_2019
synthetic["esg_2020"] = esg_2020
synthetic["esg_2021"] = esg_2021

synthetic.head()

## 5. EBITDA forecast

In [ ]:
ebitda_features = [f"ebitda_{year}" for year in range(2015, 2024)]

ebitda_result = fit_random_forest_forecast(
    dataframe=synthetic,
    feature_columns=ebitda_features,
    target_column="ebitda_2024",
    stratify_column="company_size",
)

synthetic["predicted_ebitda"] = ebitda_result["full_predictions"]

print(
    f"Synthetic-demo EBITDA MSE: "
    f"{ebitda_result['mse']:.6f}"
)

## 6. Sales Revenue forecast

In [ ]:
sales_features = [f"sales_{year}" for year in range(2015, 2024)]

sales_result = fit_random_forest_forecast(
    dataframe=synthetic,
    feature_columns=sales_features,
    target_column="sales_2024",
    stratify_column="company_size",
)

synthetic["predicted_sales"] = sales_result["full_predictions"]

print(
    f"Synthetic-demo Sales Revenue MSE: "
    f"{sales_result['mse']:.6f}"
)

## 7. Net Income forecast

In [ ]:
net_income_features = [
    f"net_income_{year}"
    for year in range(2015, 2024)
]

net_income_result = fit_random_forest_forecast(
    dataframe=synthetic,
    feature_columns=net_income_features,
    target_column="net_income_2024",
    stratify_column="company_size",
)

synthetic["predicted_net_income"] = (
    net_income_result["full_predictions"]
)

print(
    f"Synthetic-demo Net Income MSE: "
    f"{net_income_result['mse']:.6f}"
)

## 8. ESG forecast

The ESG time series is shorter than the financial histories. In the original
analysis, ESG observations for 2019–2020 were used to predict the 2021 ESG
score.

In [ ]:
esg_result = fit_random_forest_forecast(
    dataframe=synthetic,
    feature_columns=["esg_2019", "esg_2020"],
    target_column="esg_2021",
    stratify_column="company_size",
)

synthetic["predicted_esg"] = esg_result["full_predictions"]

print(
    f"Synthetic-demo ESG MSE: "
    f"{esg_result['mse']:.6f}"
)

## 9. Synthetic forecast output

The resulting table reproduces the structure required by the downstream
economic and sustainability analysis while containing only synthetic records.

In [ ]:
forecast_output = synthetic[
    [
        "company_id",
        "company_size",
        "predicted_sales",
        "predicted_ebitda",
        "predicted_net_income",
        "predicted_esg",
    ]
].copy()

forecast_output.head(10)

In [ ]:
synthetic_demo_metrics = pd.DataFrame(
    {
        "Target": [
            "Sales Revenue",
            "EBITDA",
            "Net Income",
            "ESG",
        ],
        "Synthetic Demo MSE": [
            sales_result["mse"],
            ebitda_result["mse"],
            net_income_result["mse"],
            esg_result["mse"],
        ],
    }
)

synthetic_demo_metrics

## Key Takeaways

This stage converts the model-selection exercise into target-specific company
forecasts.

The original analysis:

- trained separate Random Forest regressors for Sales Revenue, EBITDA,
  Net Income and ESG;
- used a 75/25 split stratified by company size;
- generated company-level predictions for downstream analysis;
- retained the same Random Forest configuration selected during benchmarking.

The public demonstration intentionally uses synthetic data. Its numerical
outputs are illustrative and are kept separate from the reported metrics of
the original experiment.

## Next Step

The next notebook evaluates the forecasts across company groups and translates
them into business-oriented insights, including:

- performance by company size, province and macro-sector;
- employment-growth classification;
- economic–sustainability positioning based on predicted EBITDA and ESG.